In [34]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.magicbricks.com/")

wait = WebDriverWait(driver, 15)

import time

driver.get("https://www.magicbricks.com/")
time.sleep(3)

driver.execute_script("window.scrollTo(0, 600)")
time.sleep(2)

# Scoped to the owner-properties section specifically
el = driver.find_element(By.CSS_SELECTOR, "#owner-properties a.mb-home__section__title--anchor-see-all")
driver.execute_script("arguments[0].scrollIntoView(); arguments[0].click();", el)

time.sleep(2)
print("Current URL:", driver.current_url)






Current URL: https://www.magicbricks.com/


In [35]:
# Switch to new tab
driver.switch_to.window(driver.window_handles[-1])
time.sleep(2)
print("Current URL:", driver.current_url)

# Now try scraping
owner = driver.find_elements(By.CSS_SELECTOR, ".mb-srp__card__ads--name")
price = driver.find_elements(By.CSS_SELECTOR, ".mb-srp__card__price--amount")

for o, p in zip(owner[:5], price[:5]):
    print(f"{o.text} - {p.text}")

Current URL: https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment,Penthouse,Studio-Apartment,Residential-House,Villa&cityName=Bangalore&category=B&parameter=rel&hideviewed=N&ListingsType=I&filterCount=3&incSrc=Y&fromSrc=homeSrc
Owner: Manjunath Sharma - ₹
79.6 Lac
Owner: Ahmed - ₹
2.70 Cr
Owner: Social Media Manager G - ₹
58 Lac
Owner: Devidharsan - ₹
2.50 Cr
Owner: Udit - ₹
3.65 Cr


In [36]:
time.sleep(2) 
# Scroll through the page to trigger lazy loading, then click buttons
driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
time.sleep(2)

buttons = driver.find_elements(By.CSS_SELECTOR, "div.mb-srp__card__summary__action")
for btn in buttons:
    driver.execute_script("arguments[0].click();", btn)
# Get first image (thumbnail) per card
cards = driver.find_elements(By.CSS_SELECTOR, ".mb-srp__card")

image_urls = []
for card in cards:
    try:
        img = card.find_element(By.CSS_SELECTOR, ".mb-srp__card__photo__fig--graphic")
        # data-src holds the real URL (lazy load), src may be placeholder
        url = img.get_attribute("data-src") or img.get_attribute("src")
        image_urls.append(url)
    except:
        image_urls.append(None)

for url in image_urls[:5]:
    print(url)

all_card_images = []
for card in cards:
    imgs = card.find_elements(By.CSS_SELECTOR, ".mb-srp__card__photo__fig--graphic")
    urls = [img.get_attribute("data-src") or img.get_attribute("src") for img in imgs]
    all_card_images.append(urls)


owner      = driver.find_elements(By.CSS_SELECTOR, ".mb-srp__card__ads--name")
price      = driver.find_elements(By.CSS_SELECTOR, ".mb-srp__card__price--amount")
per_sqft   = driver.find_elements(By.CSS_SELECTOR, ".mb-srp__card__price--size")
title      = driver.find_elements(By.CSS_SELECTOR, ".mb-srp__card--title")

super_area = driver.find_elements(By.CSS_SELECTOR, "[data-summary='super-area'] .mb-srp__card__summary--value")
status     = driver.find_elements(By.CSS_SELECTOR, "[data-summary='transaction'] .mb-srp__card__summary--value")
floor      = driver.find_elements(By.CSS_SELECTOR, "[data-summary='floor'] .mb-srp__card__summary--value")
facing     = driver.find_elements(By.CSS_SELECTOR, "[data-summary='facing'] .mb-srp__card__summary--value")
furnishing = driver.find_elements(By.CSS_SELECTOR, "[data-summary='furnishing'] .mb-srp__card__summary--value")


for o, p in zip(owner[:100], price[:100]):
    print(f"{o.text} - {p.text}")
    

https://img.staticmb.com/mbphoto/property/cropped_images/ver2/XIwvQlc61t8ZIpanz4mAnTVN99s9dsrotLy6tu-tbThP8bjeloKu/Photo_h470_w1080/60fc545e-5373-4a8a-9c9a-ac339ea40c7a_84658639_470_1080.jpg
https://img.staticmb.com/mbphoto/property/cropped_images/ver2/XIwvQlc61t8ZIpanz4mAnTVN99s9dcxJlofPjF6-wJmxkBru_Ytm/Photo_h470_w1080/84652205_6_PropertyImage866-5222221588226_470_1080.jpg
https://img.staticmb.com/mbphoto/property/cropped_images/ver2/XIwvQlc61t8ZIpanz4mAnTVN99s9dc5BIi1I2yNklvxE0SZUV2yn/Photo_h300_w450/84667853_4_PropertyImage423-1850622404135_300_450.jpg
https://img.staticmb.com/mbphoto/property/cropped_images/ver2/XIwvQlc61t8ZIpanz4mAnTVN99s9dc5BIi1I2yNklvxE0SZUV2yn/Photo_h300_w450/84659321_1_PropertyImage160-95383090521474_300_450.jpg
https://img.staticmb.com/mbphoto/property/cropped_images/ver2/XIwvQlc61t8ZIpanz4mAnTVN99s9dc5BIi1I2yNklvxE0SZUV2yn/Photo_h300_w450/84666367_5_hatsAppImage20260514at6.16.03PM_300_450.jpeg
Owner: Manjunath Sharma - ₹
79.6 Lac
Owner: Ahmed - ₹
2.70 Cr
Ow

In [ ]:
import pandas as pd

cards = driver.find_elements(By.CSS_SELECTOR, ".mb-srp__card")

def get_text(card, selector):
    els = card.find_elements(By.CSS_SELECTOR, selector)
    return els[0].text.strip() if els else None

rows = []
for card in cards[:100]:
    rows.append({
        'owner':      get_text(card, ".mb-srp__card__ads--name"),
        'price':      get_text(card, ".mb-srp__card__price--amount"),
        'per_sqft':   get_text(card, ".mb-srp__card__price--size"),
        'title':      get_text(card, ".mb-srp__card--title"),
        'super_area': get_text(card, "[data-summary='super-area'] .mb-srp__card__summary--value"),
        'status':     get_text(card, "[data-summary='transaction'] .mb-srp__card__summary--value"),
        'floor':      get_text(card, "[data-summary='floor'] .mb-srp__card__summary--value"),
        'facing':     get_text(card, "[data-summary='facing'] .mb-srp__card__summary--value"),
        'furnishing': get_text(card, "[data-summary='furnishing'] .mb-srp__card__summary--value"),
        'images': [img.get_attribute('data-src') or img.get_attribute('src') 
           for img in card.find_elements(By.CSS_SELECTOR, ".mb-srp__card__photo__fig--graphic")]
    })

df = pd.DataFrame(rows)
df.head()
df.to_csv('magicbricks_listings.csv', index=False)
